In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

codes = ['13', '31', '23', '29', '30', '18', '17']
# extract country code once
cntry = customer['C_PHONE'].str[:2]
# filter positive balances and desired country codes
mask = (customer['C_ACCTBAL'] > 0) & cntry.isin(codes)
# keep only needed columns and attach country code
cf = (
    customer.loc[mask, ['C_CUSTKEY', 'C_ACCTBAL']]
            .assign(CNTRYCODE=cntry[mask].values)
)
# filter balances above the group mean
mean_bal = cf['C_ACCTBAL'].mean()
cf = cf[cf['C_ACCTBAL'] > mean_bal]
# exclude customers who have any orders
orders_keys = orders['O_CUSTKEY'].unique()

customer_filtered = cf[~cf['C_CUSTKEY'].isin(orders_keys)]

# aggregate count and sum in one pass
total = (
    customer_filtered
    .groupby('CNTRYCODE', as_index=False, sort=False)
    .agg(NUMCUST=('C_CUSTKEY', 'size'),
        TOTACCTBAL=('C_ACCTBAL', 'sum'))
    .sort_values('CNTRYCODE')
)